In [1]:
import os, glob, json
import pandas as pd
PROJECT_ROOT = os.path.abspath("..")

rows = []
for p in sorted(glob.glob(os.path.join(PROJECT_ROOT, "results", "*.json"))):
    d = json.load(open(p))
    if isinstance(d, dict):
        d["file"] = os.path.basename(p)
        rows.append(d)

df = pd.DataFrame(rows)
print(df.columns.tolist())
print(df.to_string(index=False))

['model', 'val_accuracy', 'val_balanced_accuracy', 'val_macro_f1', 'mel_recall', 'mel_precision', 'best_epoch', 'file']
                              model  val_accuracy  val_balanced_accuracy  val_macro_f1  mel_recall  mel_precision  best_epoch                               file
mobilenetv2_frozen_head_fullweights      0.650783               0.597488      0.469258    0.541176       0.334545           7       baseline_mobilenet_head.json
mobilenetv2_frozen_head_sqrtweights      0.759138               0.535879      0.535118    0.488235       0.453552           7  baseline_mobilenet_head_sqrt.json
      efficientnetb0_finetuned_sqrt      0.774151               0.630767      0.609638    0.458824       0.448276          13 efficientnetb0_finetuned_sqrt.json
    efficientnetb0_frozen_head_sqrt      0.750653               0.586910      0.587597    0.441176       0.436047           7      efficientnetb0_head_sqrt.json
         mobilenetv2_finetuned_sqrt      0.780026               0.583743   

In [2]:
out = df.drop(columns=["file"]).copy()
out["backbone"] = out["model"].str.extract(r"^(mobilenetv2|efficientnetb0)")
out["stage"] = out["model"].apply(lambda m: "fine-tuned" if "finetuned" in m else "frozen head")
out["weights"] = out["model"].apply(lambda m: "full" if "fullweights" in m else "sqrt")
out["aug"] = out["model"].apply(lambda m: "v2" if "aug2" in m else "v1")
out["delta_macro_f1_vs_final"] = (out["val_macro_f1"] - 0.609638).round(3)

cols = ["model", "backbone", "stage", "weights", "aug",
        "val_accuracy", "val_balanced_accuracy", "val_macro_f1",
        "mel_recall", "mel_precision", "best_epoch", "delta_macro_f1_vs_final"]
out = out[cols].sort_values("val_macro_f1", ascending=False).round(4)

path = os.path.join(PROJECT_ROOT, "results", "experiment_log.csv")
out.to_csv(path, index=False)
print("saved:", path)
print(out.to_string(index=False))

saved: <project>\results\experiment_log.csv
                              model       backbone       stage weights aug  val_accuracy  val_balanced_accuracy  val_macro_f1  mel_recall  mel_precision  best_epoch  delta_macro_f1_vs_final
      efficientnetb0_finetuned_sqrt efficientnetb0  fine-tuned    sqrt  v1        0.7742                 0.6308        0.6096      0.4588         0.4483          13                   -0.000
    efficientnetb0_frozen_head_sqrt efficientnetb0 frozen head    sqrt  v1        0.7507                 0.5869        0.5876      0.4412         0.4360           7                   -0.022
         mobilenetv2_finetuned_sqrt    mobilenetv2  fine-tuned    sqrt  v1        0.7800                 0.5837        0.5646      0.4706         0.5369          13                   -0.045
mobilenetv2_frozen_head_sqrtweights    mobilenetv2 frozen head    sqrt  v1        0.7591                 0.5359        0.5351      0.4882         0.4536           7                   -0.075
    mo

In [3]:
import glob
for pat in ["*.npy", "*.npz", "*.csv", "*.json"]:
    for p in sorted(glob.glob(os.path.join(PROJECT_ROOT, "results", pat))):
        print(os.path.relpath(p, PROJECT_ROOT), os.path.getsize(p), "bytes")

results\experiment_log.csv 796 bytes
results\baseline_mobilenet_head.json 279 bytes
results\baseline_mobilenet_head_sqrt.json 277 bytes
results\efficientnetb0_finetuned_sqrt.json 272 bytes
results\efficientnetb0_head_sqrt.json 272 bytes
results\finetuned_mobilenet_sqrt.json 270 bytes
results\finetuned_mobilenet_sqrt_aug2.json 274 bytes


In [4]:
import sys, numpy as np
sys.path.append(PROJECT_ROOT)
from tensorflow import keras
from src.data import load_split, make_dataset, CLASSES

train_df, val_df, test_df = load_split(PROJECT_ROOT)
model = keras.models.load_model(os.path.join(PROJECT_ROOT, "models", "final_efficientnetb0.keras"))

val_probs  = model.predict(make_dataset(val_df,  batch_size=32, training=False), verbose=0)
test_probs = model.predict(make_dataset(test_df, batch_size=32, training=False), verbose=0)

np.savez(os.path.join(PROJECT_ROOT, "results", "final_model_probs.npz"),
         val_probs=val_probs, val_labels=val_df["label"].values,
         test_probs=test_probs, test_labels=test_df["label"].values)

# sanity check against the numbers in your handoff
print("shapes:", val_probs.shape, test_probs.shape)
print("test accuracy:", round(float((test_probs.argmax(1) == test_df["label"].values).mean()), 4), "(expected 0.7718)")
print("val  accuracy:", round(float((val_probs.argmax(1) == val_df["label"].values).mean()), 4), "(expected 0.7742)")

shapes: (1532, 7) (1481, 7)
test accuracy: 0.7718 (expected 0.7718)
val  accuracy: 0.7742 (expected 0.7742)


In [5]:
from sklearn.metrics import balanced_accuracy_score, f1_score

d = np.load(os.path.join(PROJECT_ROOT, "results", "final_model_probs.npz"))
probs, y = d["test_probs"], d["test_labels"]
pred = probs.argmax(1)
MEL = CLASSES.index("mel")
print("columns:", test_df.columns.tolist())
assert "lesion_id" in test_df.columns, "no lesion_id column - paste the columns line to me"

groups = test_df.reset_index(drop=True).groupby("lesion_id").indices   # lesion -> row positions
lesions = list(groups.keys())

def metrics(idx):
    yt, pp, pr = y[idx], pred[idx], probs[idx]
    out = {
        "accuracy": (yt == pp).mean(),
        "balanced_acc": balanced_accuracy_score(yt, pp),
        "macro_f1": f1_score(yt, pp, average="macro"),
    }
    flag = pr[:, MEL] >= 0.25
    is_mel = yt == MEL
    out["mel_recall@0.25"] = flag[is_mel].mean() if is_mel.any() else np.nan
    out["mel_precision@0.25"] = is_mel[flag].mean() if flag.any() else np.nan
    for c in ["df", "vasc"]:
        k = CLASSES.index(c)
        m = yt == k
        out[f"{c}_recall"] = (pp[m] == k).mean() if m.any() else np.nan
    return out

point = metrics(np.arange(len(y)))
rng = np.random.default_rng(42)
B = 1000
rows = []
for _ in range(B):
    pick = rng.choice(len(lesions), size=len(lesions), replace=True)
    idx = np.concatenate([groups[lesions[i]] for i in pick])
    rows.append(metrics(idx))
boot = pd.DataFrame(rows)

print(f"test images {len(y)}, lesions {len(lesions)}, resamples {B}\n")
for k in point:
    lo, hi = np.nanpercentile(boot[k], [2.5, 97.5])
    print(f"{k:20s} {point[k]:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

columns: ['lesion_id', 'image_id', 'dx', 'split', 'path', 'abs_path', 'label']
test images 1481, lesions 1121, resamples 1000

accuracy             0.772   95% CI [0.746, 0.797]
balanced_acc         0.615   95% CI [0.555, 0.674]
macro_f1             0.587   95% CI [0.525, 0.637]
mel_recall@0.25      0.612   95% CI [0.527, 0.699]
mel_precision@0.25   0.331   95% CI [0.262, 0.401]
df_recall            0.450   95% CI [0.214, 0.737]
vasc_recall          0.789   95% CI [0.565, 0.955]


In [6]:
ci = pd.DataFrame({
    "metric": list(point.keys()),
    "estimate": [point[k] for k in point],
    "ci_low":  [np.nanpercentile(boot[k], 2.5)  for k in point],
    "ci_high": [np.nanpercentile(boot[k], 97.5) for k in point],
}).round(4)
ci.to_csv(os.path.join(PROJECT_ROOT, "results", "test_bootstrap_ci.csv"), index=False)
print(ci.to_string(index=False))

            metric  estimate  ci_low  ci_high
          accuracy    0.7718  0.7461   0.7971
      balanced_acc    0.6149  0.5546   0.6740
          macro_f1    0.5872  0.5246   0.6373
   mel_recall@0.25    0.6121  0.5273   0.6989
mel_precision@0.25    0.3311  0.2618   0.4006
         df_recall    0.4500  0.2142   0.7372
       vasc_recall    0.7895  0.5650   0.9546


In [7]:
full = pd.concat([train_df, val_df, test_df], ignore_index=True)

n_per_lesion = full.groupby("lesion_id")["image_id"].transform("count")
full["multi"] = n_per_lesion > 1

summary = full.groupby("dx").agg(
    images=("image_id", "count"),
    lesions=("lesion_id", "nunique"),
    imgs_in_multi=("multi", "sum"),
)
summary["pct_imgs_in_multi"] = (100 * summary["imgs_in_multi"] / summary["images"]).round(1)
summary["max_imgs_per_lesion"] = full.groupby("dx").apply(
    lambda g: g.groupby("lesion_id").size().max())

print(summary.to_string())
print("\nall classes:", len(full), "images,", full["lesion_id"].nunique(), "lesions,",
      round(100 * full["multi"].mean(), 1), "% of images in multi-image lesions")

       images  lesions  imgs_in_multi  pct_imgs_in_multi  max_imgs_per_lesion
dx                                                                           
akiec     327      228            176               53.8                    4
bcc       514      327            339               66.0                    4
bkl      1099      727            659               60.0                    6
df        115       73             76               66.1                    3
mel      1113      614            883               79.3                    6
nv       6705     5403           2290               34.2                    6
vasc      142       98             78               54.9                    4

all classes: 10015 images, 7470 lesions, 44.9 % of images in multi-image lesions


In [8]:
from sklearn.model_selection import train_test_split

full = full.reset_index(drop=True)
dx = full["dx"].values
lesion = full["lesion_id"].values

def sim(seed):
    idx = np.arange(len(full))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=dx, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=dx[tmp], random_state=seed)
    train_lesions = set(lesion[tr])
    leaked = np.array([l in train_lesions for l in lesion[te]])
    return dx[te], leaked

rows = []
for seed in range(20):
    te_dx, leaked = sim(seed)
    d = pd.DataFrame({"dx": te_dx, "leaked": leaked})
    r = d.groupby("dx")["leaked"].mean() * 100
    r["ALL"] = leaked.mean() * 100
    rows.append(r)

res = pd.DataFrame(rows)
out = pd.DataFrame({"mean_%": res.mean().round(1), "min_%": res.min().round(1), "max_%": res.max().round(1)})
print("% of test images whose lesion also appears in train (image-level split, 20 random seeds)")
print(out.to_string())

# sanity check on your real split_v1 (lesion-level): should be 0
print("\nreal split: test lesions also in train =", len(set(test_df["lesion_id"]) & set(train_df["lesion_id"])))

% of test images whose lesion also appears in train (image-level split, 20 random seeds)
       mean_%  min_%  max_%
dx                         
akiec    41.7   30.6   55.1
bcc      48.5   36.4   61.0
bkl      46.5   41.2   53.9
df       50.9   23.5   76.5
mel      61.8   51.5   70.7
nv       26.8   25.0   28.6
vasc     44.8   27.3   59.1
ALL      35.0   33.1   36.9

real split: test lesions also in train = 0


In [9]:
out.to_csv(os.path.join(PROJECT_ROOT, "results", "leakage_exposure_image_level_split.csv"))
print("saved")

saved


In [10]:
d = np.load(os.path.join(PROJECT_ROOT, "results", "final_model_probs.npz"))
vp, vy = d["val_probs"], d["val_labels"]
MEL = CLASSES.index("mel")
is_mel = vy == MEL
n_mel = int(is_mel.sum())

rows = []
for t in np.round(np.arange(0.05, 0.65, 0.05), 2):
    flag = vp[:, MEL] >= t
    n_flag = int(flag.sum())
    tp = int((flag & is_mel).sum())
    rec = tp / n_mel
    prec = tp / n_flag if n_flag else np.nan
    f1 = 2 * prec * rec / (prec + rec) if n_flag and (prec + rec) > 0 else np.nan
    f2 = 5 * prec * rec / (4 * prec + rec) if n_flag and (4 * prec + rec) > 0 else np.nan
    rows.append({"threshold": t, "flagged": n_flag, "flagged_%": round(100 * n_flag / len(vy), 1),
                 "mel_found": tp, "recall": rec, "precision": prec, "F1": f1, "F2": f2})

sweep = pd.DataFrame(rows).round(3)
sweep.to_csv(os.path.join(PROJECT_ROOT, "results", "val_mel_threshold_sweep.csv"), index=False)
print(f"validation: {len(vy)} images, {n_mel} melanomas")
print(sweep.to_string(index=False))

validation: 1532 images, 170 melanomas
 threshold  flagged  flagged_%  mel_found  recall  precision    F1    F2
      0.05      789       51.5        154   0.906      0.195 0.321 0.524
      0.10      617       40.3        142   0.835      0.230 0.361 0.547
      0.15      489       31.9        124   0.729      0.254 0.376 0.530
      0.20      401       26.2        117   0.688      0.292 0.410 0.541
      0.25      316       20.6        102   0.600      0.323 0.420 0.512
      0.30      267       17.4         96   0.565      0.360 0.439 0.507
      0.35      219       14.3         87   0.512      0.397 0.447 0.484
      0.40      175       11.4         77   0.453      0.440 0.446 0.450
      0.45      149        9.7         73   0.429      0.490 0.458 0.440
      0.50      127        8.3         66   0.388      0.520 0.444 0.409
      0.55      101        6.6         54   0.318      0.535 0.399 0.346
      0.60       81        5.3         48   0.282      0.593 0.382 0.315
